In [ ]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display


In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

In [ ]:

openai = OpenAI()

openrouter_url = "https://openrouter.ai/api/v1"

openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)

In [ ]:
models = ["openai/gpt-5.1-codex", "anthropic/claude-haiku-4.5", "gemini-2.5-flash", "openai/gpt-oss-120b"]

clients = {
    "openai/gpt-5.1-codex": openrouter,
    "anthropic/claude-haiku-4.5": openrouter,
    "gemini-2.5-flash": openrouter,
    "openai/gpt-oss-120b": openrouter
}

In [ ]:


system_prompt = f""" You are a python testing expert.
Your task is to create Python  test code for the python code provided by the user. The test code should be written in python and should be able to be run using unittest. The test code should cover all the functions in the provided python code and should include edge cases. The test code should be written in a way that it can be easily understood by other developers. The test code should be written in a way that it can be easily maintained and updated as the provided python code changes. The test code should be written in a way that it can be easily integrated into a CI/CD pipeline.
Respond only with  code. Do not provide any explanation other than occasional comments.
"""

def user_prompt_for(python):
    return f"""
Create test code for the following python code:
```python
{python}
```
"""

In [ ]:
def messages_for(python, model):
    if model.startswith("openai/"):
        from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam
        return [
            ChatCompletionSystemMessageParam(role="system", content=system_prompt),
            ChatCompletionUserMessageParam(role="user", content=user_prompt_for(python))
        ]
    else:
        return [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt_for(python)}
        ]


In [ ]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value

def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [ ]:
def port(model, python):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python, model), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```rust','').replace('```','')
    return reply

In [ ]:
def run_python(code):
    import types
    import unittest
    globals_dict = {"__builtins__": __builtins__}
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer
    try:
        exec(code, globals_dict)
        # If unittest is present, run tests
        if any('unittest' in line for line in code.splitlines()):
            # Find all TestCase classes in globals_dict
            test_cases = [obj for obj in globals_dict.values()
                          if isinstance(obj, type) and issubclass(obj, unittest.TestCase)]
            if test_cases:
                suite = unittest.TestSuite()
                for case in test_cases:
                    suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(case))
                runner = unittest.TextTestRunner(stream=buffer, verbosity=2)
                runner.run(suite)
            else:
                buffer.write("No unittest.TestCase classes found in test code.\n")
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [ ]:
from styles import CSS

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Create test Python") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (original)",
                value=python_hard,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            python_test = gr.Code(
                label=f"Test code (generated)",
                value="",
                language="python",
                lines=26
            )

    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value=models[0], show_label=False)
        python_run_test = gr.Button(f"Run test code", elem_classes=["convert-btn"])
        convert = gr.Button(f"Generate test code", elem_classes=["convert-btn"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            python_test_out = gr.TextArea(label="Python test result", lines=8, elem_classes=["py-out"])
            # cpp_out = gr.TextArea(label=f"Python test result", lines=8, elem_classes=["py-out"])

    convert.click(fn=port, inputs=[model, python], outputs=[python_test])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    python_run_test.click(fn=run_python, inputs=[python_test], outputs=[python_test_out])

ui.launch(inbrowser=True)